# Demonstração ponta a ponta

Executa um experimento do `agentic-tabular-pipeline` a partir de uma configuração YAML
e inspeciona o ranking de modelos, os eventos agentivos e o relatório técnico.

**Pré-requisitos**

1. Dependências instaladas: `pip install -e .[dev]`
2. Base de demonstração gerada: `python -m scripts.generate_demo_dataset`
3. *(Opcional)* PostgreSQL acessível via `DATABASE_URL` para persistir e consultar a trilha de auditoria (seção 23). Sem ele, o pipeline roda em memória e inspecionamos o resultado retornado.

## 0. Preparar o ambiente

Garante que o notebook execute a partir da raiz do repositório (para que os caminhos relativos da configuração resolvam).

In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")
print("Diretório de trabalho:", Path.cwd())
print("DATABASE_URL definido:", bool(os.environ.get("DATABASE_URL")))

## 1. Carregar a configuração do experimento

Para a demonstração reduzimos o conjunto de modelos e as épocas do autoencoder, mantendo a execução rápida.

In [ ]:
from src.pipelines.training import load_config

config = load_config("configs/experiment_example.yaml")
config["models"]["include"] = ["logistic_regression", "random_forest", "hist_gradient_boosting"]
config["autoencoder"]["epochs"] = 10
config["experiment"]

## 2. Executar o pipeline

Encadeia os agentes: perfilamento → limpeza → engenharia de atributos → split → treino cruzado → autoencoder → avaliação → relatório.

Se `DATABASE_URL` estiver definido, tudo é persistido no PostgreSQL; caso contrário, roda em memória.

In [ ]:
from src.pipelines.training import run_experiment, _build_recorder

recorder = _build_recorder(config)  # usa DATABASE_URL se disponível, senão None
event_sink = recorder.event_sink if recorder is not None else None

result = run_experiment(config, recorder=recorder, event_sink=event_sink)

print("Persistência:", "PostgreSQL" if recorder else "em memória (defina DATABASE_URL para gravar)")
print("Modelo recomendado:", result["best_model"])
result["metrics"]

## 3. Ranking de modelos (RF11)

Métrica primária por fold: média, desvio e número de folds.

In [ ]:
import pandas as pd

pd.DataFrame([
    {
        "modelo": e["model_name"],
        "média": round(e["primary_mean"], 4),
        "desvio": round(e.get("primary_std") or 0.0, 4),
        "folds": e["n_folds"],
        "tempo_fit_s": round(e.get("mean_fit_seconds") or 0.0, 3),
    }
    for e in result["ranking"]
])

## 4. Histórico agentivo — trilha de auditoria (RNF03, seção 23)

Consulta SQL equivalente à da seção 23 do documento. Requer PostgreSQL.

In [ ]:
if recorder is not None:
    from src.db.models import get_engine

    engine = get_engine()
    events = pd.read_sql(
        "SELECT created_at, agent_name, event_type, rationale "
        "FROM agent_events WHERE experiment_id = %(eid)s ORDER BY created_at",
        engine, params={"eid": str(result["experiment_id"])},
    )
    display(events)
else:
    print("DATABASE_URL não definido — execute com PostgreSQL para a trilha de auditoria.")
    print("Em memória, as justificativas dos agentes estão em result[<etapa>].")

## 5. Ranking agregado por SQL/JSONB (seção 23)

Demonstra a agregação do ranking diretamente sobre o campo `metrics_json` (JSONB).

In [ ]:
if recorder is not None:
    metric = config["experiment"]["primary_metric"]
    ranking_sql = pd.read_sql(
        f"SELECT model_name, "
        f"round(AVG((metrics_json->>'{metric}')::numeric), 4) AS mean_metric, "
        f"round(STDDEV((metrics_json->>'{metric}')::numeric), 4) AS std_metric, "
        f"COUNT(*) AS folds "
        f"FROM model_results WHERE run_id = %(rid)s AND fold IS NOT NULL "
        f"GROUP BY model_name ORDER BY mean_metric DESC",
        engine, params={"rid": str(result["run_id"])},
    )
    display(ranking_sql)
else:
    print("DATABASE_URL não definido — consulta SQL indisponível em modo memória.")

## 6. Relatório técnico (RF14)

Relatório gerado automaticamente, com metodologia, resultados, explicabilidade, riscos e limitações.

In [ ]:
from IPython.display import Markdown

Markdown(result["report_markdown"])